# E01 Readiness and Launch Notebook

This notebook turns the E01 operator checklist into a concrete workflow.

## Purpose

E01 is the **target granularity experiment**. The goal is to compare:

- `emotion`
- `coarse_affect`
- `binary_valence_like`

under the fixed core setup:

- model: `ridge`
- primary CV: `within_subject_loso_session`

## Recommended execution policy

Use the **registry path**, not the workbook path, as the source of truth for execution.

## Workflow

1. Freeze the exact E01 definition
2. Run a dry-run of E01
3. Audit dataset sufficiency by target / subject / session
4. Verify index and feature-cache integrity
5. Run one smoke variant
6. Launch the full E01 primary batch
7. Review outputs and make the D01 decision
8. Decide whether to add the secondary transfer check


## Notes

- The notebook assumes you run it from the **repository root**.
- Command cells use `python -m uv run ...` to match your project style.
- The audit cells are designed to be helpful even before a full run.
- Some cells are intentionally conservative and may need small path edits if your local environment differs.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from typing import Iterable

import pandas as pd

REPO_ROOT = Path(".").resolve()
INDEX_CSV = REPO_ROOT / "Data/processed/dataset_index.csv"
DATA_ROOT = REPO_ROOT / "Data"
CACHE_DIR = REPO_ROOT / "Data/processed/feature_cache"
REGISTRY = REPO_ROOT / "configs/decision_support_registry.json"
OUTPUT_ROOT = REPO_ROOT / "outputs/artifacts/decision_support"
EXPERIMENT_ID = "E01"

print("Repo root:", REPO_ROOT)
print("Index exists:", INDEX_CSV.exists(), INDEX_CSV)
print("Data root exists:", DATA_ROOT.exists(), DATA_ROOT)
print("Cache dir exists:", CACHE_DIR.exists(), CACHE_DIR)
print("Registry exists:", REGISTRY.exists(), REGISTRY)
print("Output root:", OUTPUT_ROOT)


## 1. Freeze the exact E01 definition

This cell checks the registry entry so you confirm E01 means exactly what you intend before running anything.


In [ ]:
registry_data = json.loads(REGISTRY.read_text(encoding="utf-8"))
registry_data.keys()


In [ ]:
# Try to locate the E01 entry in a reasonably flexible way.
e01_entry = None

if isinstance(registry_data, dict):
    if "experiments" in registry_data and isinstance(registry_data["experiments"], list):
        for item in registry_data["experiments"]:
            if isinstance(item, dict) and item.get("experiment_id") == EXPERIMENT_ID:
                e01_entry = item
                break
    elif EXPERIMENT_ID in registry_data:
        e01_entry = registry_data[EXPERIMENT_ID]

if e01_entry is None:
    print("Could not automatically locate E01 in the registry. Inspect the printed structure manually.")
else:
    from pprint import pprint
    pprint(e01_entry)


## 2. Dry-run E01

Expected outcome:

- E01 resolves successfully
- it expands into concrete variants
- the variants use:
  - `ridge`
  - `within_subject_loso_session`
  - one of the three target definitions

This cell prints the command first, then runs it.


In [ ]:
dry_run_cmd = [
    sys.executable, "-m", "uv", "run",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", EXPERIMENT_ID,
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(OUTPUT_ROOT),
    "--dry-run",
]
print(" ".join(f'"{part}"' if " " in part else part for part in dry_run_cmd))


In [ ]:
# Uncomment to actually execute the dry-run.
# result = subprocess.run(dry_run_cmd, text=True, capture_output=True)
# print("Return code:", result.returncode)
# print(result.stdout)
# print(result.stderr)


## 3. Dataset sufficiency audit

This is the most important scientific gate before the full E01 launch.

The audit below checks, for each candidate target:
- overall class counts
- per-subject class counts
- per-subject/per-session class coverage

It helps answer:
- is `emotion` too sparse?
- is `binary_valence_like` too easy but too reductive?
- is `coarse_affect` the best compromise?


In [ ]:
df = pd.read_csv(INDEX_CSV)
print(df.shape)
print(df.columns.tolist())
df.head()


In [ ]:
required_base_cols = ["subject", "session"]
missing_base = [c for c in required_base_cols if c not in df.columns]
if missing_base:
    raise ValueError(f"Missing required columns for the audit: {missing_base}")

candidate_targets = [c for c in ["emotion", "coarse_affect", "binary_valence_like"] if c in df.columns]
print("Candidate targets found:", candidate_targets)


In [ ]:
def summarize_target(df: pd.DataFrame, target: str) -> dict:
    out = {}
    valid = df.dropna(subset=[target]).copy()
    out["n_rows"] = len(valid)
    out["n_subjects"] = valid["subject"].nunique()
    out["n_sessions"] = valid["session"].nunique()
    out["class_counts"] = valid[target].value_counts(dropna=False).to_dict()
    out["classes_per_subject"] = (
        valid.groupby("subject")[target]
        .nunique()
        .sort_values()
        .to_dict()
    )
    out["classes_per_subject_session"] = (
        valid.groupby(["subject", "session"])[target]
        .nunique()
        .sort_values()
        .to_dict()
    )
    return out

audit = {target: summarize_target(df, target) for target in candidate_targets}
audit


In [ ]:
for target, info in audit.items():
    print("=" * 90)
    print("TARGET:", target)
    print("Rows:", info["n_rows"])
    print("Subjects:", info["n_subjects"])
    print("Sessions:", info["n_sessions"])
    print("Class counts:")
    print(pd.Series(info["class_counts"]).sort_values(ascending=False))
    print()
    cps = pd.Series(info["classes_per_subject"]).sort_values()
    print("Number of unique classes per subject:")
    print(cps)
    print()
    cpss = pd.Series(info["classes_per_subject_session"]).sort_values()
    print("Number of unique classes per subject-session (smallest first):")
    print(cpss.head(20))
    print()


### LOSO session viability check

For each target and subject, this estimates whether session-level leave-one-session-out looks viable in a basic sense.

This is not a full proof of modeling viability, but it catches obvious cases where held-out sessions may be missing class coverage.


In [ ]:
def loso_session_viability(df: pd.DataFrame, target: str) -> pd.DataFrame:
    rows = []
    valid = df.dropna(subset=[target]).copy()
    for subject, sdf in valid.groupby("subject"):
        all_classes = set(sdf[target].dropna().unique().tolist())
        sessions = sorted(sdf["session"].dropna().unique().tolist())
        for heldout_session in sessions:
            train = sdf[sdf["session"] != heldout_session]
            test = sdf[sdf["session"] == heldout_session]
            train_classes = set(train[target].dropna().unique().tolist())
            test_classes = set(test[target].dropna().unique().tolist())
            rows.append({
                "target": target,
                "subject": subject,
                "heldout_session": heldout_session,
                "n_train": len(train),
                "n_test": len(test),
                "n_all_classes": len(all_classes),
                "n_train_classes": len(train_classes),
                "n_test_classes": len(test_classes),
                "missing_from_train": sorted(test_classes - train_classes),
                "missing_from_test": sorted(all_classes - test_classes),
                "train_has_all_global_classes": train_classes == all_classes,
            })
    return pd.DataFrame(rows)

viability_frames = []
for target in candidate_targets:
    viability_frames.append(loso_session_viability(df, target))

viability = pd.concat(viability_frames, ignore_index=True) if viability_frames else pd.DataFrame()
viability.head()


In [ ]:
if not viability.empty:
    display_cols = [
        "target", "subject", "heldout_session",
        "n_train", "n_test",
        "n_all_classes", "n_train_classes", "n_test_classes",
        "train_has_all_global_classes", "missing_from_train"
    ]
    problems = viability[
        (~viability["train_has_all_global_classes"]) |
        (viability["n_test"] == 0) |
        (viability["n_train"] == 0)
    ][display_cols].copy()
    print("Potential LOSO viability problems:")
    display(problems if len(problems) else pd.DataFrame(columns=display_cols))


## 4. Feature-cache and index integrity checks

These checks are meant to catch avoidable I/O and cache issues before a real run.


In [ ]:
print("Index exists:", INDEX_CSV.exists())
print("Cache dir exists:", CACHE_DIR.exists())
if CACHE_DIR.exists():
    npz_files = list(CACHE_DIR.rglob("*.npz"))
    print("Number of .npz cache files found:", len(npz_files))
    print("First few cache files:")
    for p in npz_files[:10]:
        print(" -", p)


In [ ]:
# Light sanity check for subjects in index
if "subject" in df.columns:
    print("Subjects:", sorted(df["subject"].dropna().unique().tolist())[:20])
    print("Number of subjects:", df["subject"].nunique())


## 5. Run one smoke variant first

Recommended first smoke case:
- one subject
- target = `coarse_affect`
- model = `ridge`
- CV = `within_subject_loso_session`

Edit `SMOKE_SUBJECT` if needed.


In [ ]:
SMOKE_SUBJECT = sorted(df["subject"].dropna().unique().tolist())[0] if "subject" in df.columns and len(df) else "sub-001"
RUN_ID = f"exploratory_{SMOKE_SUBJECT}_ridge_coarse_affect_e01_smoke"

smoke_cmd = [
    "thesisml-run-experiment",
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--target", "coarse_affect",
    "--model", "ridge",
    "--cv", "within_subject_loso_session",
    "--subject", str(SMOKE_SUBJECT),
    "--run-id", RUN_ID,
]
print(" ".join(f'"{part}"' if " " in part else part for part in smoke_cmd))


In [ ]:
# Uncomment to actually execute the smoke run.
# result = subprocess.run(smoke_cmd, text=True, capture_output=True)
# print("Return code:", result.returncode)
# print(result.stdout)
# print(result.stderr)


## 6. Launch the full primary E01 batch

Only do this after:
- the dry-run passes
- the sufficiency audit looks acceptable
- the smoke variant succeeds

This launches the full primary E01 batch from the registry path.


In [ ]:
full_cmd = [
    sys.executable, "-m", "uv", "run",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", EXPERIMENT_ID,
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(OUTPUT_ROOT),
]
print(" ".join(f'"{part}"' if " " in part else part for part in full_cmd))


In [ ]:
# Uncomment to actually execute the full E01 batch.
# result = subprocess.run(full_cmd, text=True, capture_output=True)
# print("Return code:", result.returncode)
# print(result.stdout)
# print(result.stderr)


## 7. Post-run review guide

Do **not** choose the target only by one metric.

For each target, review:
- balanced accuracy
- macro-F1
- class-wise recall
- per-subject stability
- fold validity
- whether performance is inflated by easier target definition rather than better construct fit

Suggested decision rule:
- reject targets that are not data-sufficient
- prefer stable targets across subjects and sessions
- prefer targets that preserve thesis construct validity rather than merely maximizing raw ease of classification


## 8. Secondary transfer check decision

The workbook narrative mentions a secondary frozen-transfer check, while the executable registry may only implement the primary within-subject comparison.

After the primary E01 batch, explicitly decide one of these:
- E01 is only the within-subject target-lock experiment
- E01 requires a secondary cross-person transfer follow-up before D01 is locked


## 9. Write-back checklist

After E01 completes, update:
- Decision_Log for D01
- Run_Log
- Trial_Results
- optionally Data_Profile with class-count / sufficiency evidence

Record:
- winning target
- why it won
- why the alternatives did not
- any tradeoff between learnability and construct validity
- whether secondary transfer remains pending


## Optional next step

After this notebook, the most useful follow-up would be a small report notebook or script that aggregates the completed E01 artifacts into a target-by-subject comparison table and a D01 decision memo draft.
